# Figure 5

This notebook reproduces the original Figure 5 plot family, now using disjoint cross-fold runs.

Statistical reporting:
- Per-fish: two-sided exact sign-count tests across condition-cell values within each fish, reported separately for diagonal and off-diagonal train/test pairs
- Pooled: two-sided exact sign-count tests across all condition-cell values, reported separately for diagonal and off-diagonal train/test pairs


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from probabilistic_model_synthesis.visualization import import_style
import_style()

from cross_fold_analysis import build_elbo_arrays
from cross_fold_analysis import discover_layout
from cross_fold_analysis import load_results
from cross_fold_analysis import make_heatmaps
from cross_fold_analysis import make_per_fish_improvement_heatmaps
from cross_fold_analysis import per_fish_pair_type_tests
from cross_fold_analysis import pooled_pair_type_tests


## Config


In [ ]:
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'probabilistic_model_synthesis').is_dir() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

# Top-level directory holding paired cross-fold results.
base_dir = REPO_ROOT / 'results/publication_results/gnldr/quantification_paired'

# Name used for all files containing post-processed results.
pp_file = 'pp_fit_results.pt'

# Type of model fit to evaluate: 'ip' or 'sp'
mdl_type = 'ip'

# Save folder for output figures.
save_folder = base_dir / 'imgs'
os.makedirs(save_folder, exist_ok=True)


## Discover layout and load results


In [ ]:
layout = discover_layout(base_dir)
train_conds = layout['train_conds']
fold_names = layout['fold_names']
test_subjs = layout['test_subjs']

print('train_conds:', train_conds)
print('fold_names:', fold_names)
print('test_subjs:', test_subjs)


In [ ]:
rs = load_results(
    base_dir=base_dir,
    train_conds=train_conds,
    fold_names=fold_names,
    test_subjs=test_subjs,
    fit_types=['multi_cond', 'single_cond'],
    pp_file=pp_file,
)

arrs = build_elbo_arrays(
    rs=rs,
    train_conds=train_conds,
    fold_names=fold_names,
    test_subjs=test_subjs,
    mdl_type=mdl_type,
    test_periods=['omr_forward', 'omr_right', 'omr_left'],
)

sb = arrs['sb']
db = arrs['db']
delta = arrs['delta']
fish_fold_diag = arrs['fish_fold_diag']
fish_fold_offdiag = arrs['fish_fold_offdiag']
fish_fold_all = arrs['fish_fold_all']

print('delta array shape:', delta.shape, '(fish, fold, train, test)')


## Statistics


In [ ]:
per_fish_df = per_fish_pair_type_tests(
    delta=delta,
    diag_mask=arrs['diag_mask'],
    offdiag_mask=arrs['offdiag_mask'],
    fish_ids=test_subjs,
)
print('Per-fish two-sided exact sign-count tests across condition-cell values, diagonal and off-diagonal separately:')
display(per_fish_df)

pooled_df = pooled_pair_type_tests(
    delta=delta,
    diag_mask=arrs['diag_mask'],
    offdiag_mask=arrs['offdiag_mask'],
    fish_ids=test_subjs,
)
print('Pooled two-sided exact sign-count tests across condition-cell values, diagonal and off-diagonal separately:')
display(pooled_df)


## Cross-Fold Summary Plots


### Per-fish ΔELBO heatmaps


In [ ]:
_ = make_per_fish_improvement_heatmaps(
    delta=delta,
    fish_ids=test_subjs,
    save_dir=save_folder,
    save_prefix='elbo_improvements_subj_',
)
print('Saved per-fish heatmaps to', save_folder)


In [ ]:
import matplotlib.colors as mcolors

cond_labels = ['F', 'R', 'L']
fish_mean_delta = np.nanmean(delta, axis=1)

vmax = np.nanmax(np.abs(fish_mean_delta))
vmax = float(vmax if np.isfinite(vmax) and vmax > 0 else 1.0)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

fig, axs = plt.subplots(
    1,
    len(test_subjs),
    figsize=(5, 2.4),
    sharey=True,
    constrained_layout=False,
    gridspec_kw={'wspace': -0.05},
)
# fig.subplots_adjust(left=0.08, right=0.99, bottom=0.34, top=0.86)
axs = np.atleast_1d(axs)

im = None
for fi, (ax, fish) in enumerate(zip(axs, test_subjs)):
    im = ax.imshow(fish_mean_delta[fi], cmap='bwr', norm=norm)
    ax.set_xticks(np.arange(3), labels=cond_labels)
    ax.set_yticks(np.arange(3), labels=cond_labels)
    ax.set_xlabel('Test condition')
    if fi == 0:
        ax.set_ylabel('Target fish \ntrain condition')
    else:
        ax.set_ylabel('')
        ax.tick_params(labelleft=False)
    ax.set_title(f'Fish {fi + 1}', fontsize=7)
    for r in range(3):
        for c in range(3):
            ax.text(
                c,
                r,
                f'{fish_mean_delta[fi, r, c]:.0f}',
                ha='center',
                va='center',
                fontsize=5,
                color='black',
            )
    # ax.spines['top'].set_visible(False)
    # ax.spines['right'].set_visible(False)

cbar = fig.colorbar(
    im,
    ax=axs,
    orientation='horizontal',
    fraction=0.05,
    pad=0.3,
    aspect=22,
)
cbar.set_ticks([-15000, 0, 15000])
cbar.set_label('Improvement by DPMS: $\\Delta$ELBO (DB - SB)')
cbar.ax.xaxis.set_label_position('top')
cbar.ax.xaxis.set_ticks_position('bottom')

save_path = save_folder / 'elbo_improvements_per_fish_heatmaps_shared_colorbar.svg'
fig.savefig(
    save_path,
    format='svg',
    dpi=500,
    bbox_inches='tight',
    pad_inches=0.05,
    transparent=True,
)


### Mean and positive-count heatmaps


In [ ]:
fig_mean2, _, fig_cnt2, _ = make_heatmaps(
    delta=delta,
    save_mean_path=save_folder / 'elbo_mean_delta_heatmap_cross_fold.svg',
    save_count_path=save_folder / 'elbo_positive_count_heatmap_cross_fold.svg',
)
fig_mean2
fig_cnt2
